## Imports :

In [1]:
import joblib
import copy

import scipy as sp
from scipy.stats import skew, boxcox, mode
from scipy.spatial import distance
import cvxpy as cp

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import missingno as msno
from sklearn.impute import SimpleImputer 

from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, ElasticNet, Lasso, RidgeCV, LassoCV, ElasticNetCV, LogisticRegression, SGDClassifier, LogisticRegressionCV, RidgeClassifier, RidgeClassifierCV
from sklearn.preprocessing import PolynomialFeatures, SplineTransformer, PowerTransformer, FunctionTransformer, QuantileTransformer, LabelEncoder, OneHotEncoder, MinMaxScaler, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn import metrics
from sklearn.utils.extmath import weighted_mode
from sklearn.model_selection import train_test_split, KFold, LeaveOneOut, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.datasets import load_breast_cancer, make_classification, make_blobs, make_circles, load_iris, make_regression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.svm import LinearSVC, SVC, NuSVC
from sklearn.neighbors import KDTree, BallTree, NearestNeighbors, KNeighborsClassifier, KNeighborsRegressor
from sklearn.tree import *
from sklearn.ensemble import *
from xgboost import *
from lightgbm import *
from catboost import *

from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.basemap import Basemap
from scipy.ndimage import gaussian_filter1d
import seaborn as sns
from mlxtend.plotting import plot_decision_regions

from mlxtend.evaluate import bias_variance_decomp

## Load data :

In [2]:
dfN = pd.read_csv('../data/train_set_eN.csv')
dfS = pd.read_csv('../data/train_set_eS.csv')

In [3]:
x_trainN, y_trainN = np.array(dfN.iloc[:, :-1]), np.array(dfN.iloc[:, [-1]])
x_trainN.shape, y_trainN.shape

((28932, 31), (28932, 1))

In [4]:
x_trainS, y_trainS = np.array(dfS.iloc[:, :-1]), np.array(dfS.iloc[:, [-1]])
x_trainS.shape, y_trainS.shape

((28932, 31), (28932, 1))

In [5]:
df_vallN = pd.read_csv("../data/vall_set_eN.csv")
df_vallS = pd.read_csv("../data/vall_set_eS.csv")

In [6]:
x_vallN, y_vallN = np.array(df_vallN.iloc[:, :-1]), np.array(df_vallN.iloc[:, [-1]])
x_vallN.shape, y_vallN.shape

((7234, 31), (7234, 1))

In [7]:
x_vallS, y_vallS = np.array(df_vallS.iloc[:, :-1]), np.array(df_vallS.iloc[:, [-1]])
x_vallS.shape, y_vallS.shape

((7234, 31), (7234, 1))

In [8]:
df_testN = pd.read_csv("../data/test_set_eN.csv")
df_testS = pd.read_csv("../data/test_set_eS.csv")

In [9]:
x_testN, y_testN = np.array(df_testN.iloc[:, :-1]), np.array(df_testN.iloc[:, [-1]])
x_testN.shape, y_testN.shape

((9043, 31), (9043, 1))

In [10]:
x_testS, y_testS = np.array(df_testS.iloc[:, :-1]), np.array(df_testS.iloc[:, [-1]])
x_testS.shape, y_testS.shape

((9043, 31), (9043, 1))

## Train :

In [15]:
model_logistic = joblib.load('best_logistic_model')
model_mlp = joblib.load('best_MLPnetwork')
model_bagging = joblib.load('best_Bagging')

estimators = [model_mlp, model_logistic, model_bagging]

In [19]:
model1 = VotingClassifier(estimators=[('bagging', model_bagging), ('logistic', model_logistic), ('mlp', model_mlp)], voting='soft')
model1.fit(x_trainS, y_trainS.ravel())

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('bagging', ...), ('logistic', ...), ...]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1


In [20]:
model1.score(x_trainS, y_trainS), model1.score(x_vallS, y_vallS)

(0.9344324623254527, 0.9032347249101466)

In [21]:
tn, fp, fn, tp = metrics.confusion_matrix(y_trainS, model1.predict(x_trainS)).ravel().tolist()
tn, fp, fn, tp

(24430, 1117, 780, 2605)

In [22]:
tn, fp, fn, tp = metrics.confusion_matrix(y_vallS, model1.predict(x_vallS)).ravel().tolist()
tn, fp, fn, tp

(5990, 398, 302, 544)